# Part 1: Getting started with GROQ API

In [ ]:
import os
from dotenv import load_dotenv
from groq import Groq

## Task 1: Groq API setup and Basic Chat

In [ ]:
load_dotenv()

In [ ]:
api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY is not set in the .env file.")

In [ ]:
client = Groq(api_key=api_key)

In [ ]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": "What is Generative AI? Explain in simple terms."
        }
    ]
)
print(response.choices[0].message.content)

## Task 2: Build a Chatbot (Core Logic)

In [ ]:

client = Groq(api_key=os.getenv("GROQ_API_KEY"))


In [ ]:
MODEL_NAME = "llama-3.3-70b-versatile"

In [ ]:
def groq_chat(prompt: str):
    """
    Send a user prompt to Groq and return the model response.
    """
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful AI assistant. "
                    "Give clear, accurate, and concise answers."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

In [ ]:
questions = [
        "What is Python?",
        "Explain REST API in simple terms.",
        "What is RAG in Generative AI?",
        "What is the difference between AI and Machine Learning?",
        "Explain vector databases."
    ]

for question in questions:
    response = groq_chat(question)
    print(f"Assistant: {response}")

# Part 2: Groq + RAG

## Task 3: Groq based RAG Pipeline

In [ ]:
import os
import faiss
from dotenv import load_dotenv
from groq import Groq
from sentence_transformers import SentenceTransformer

In [ ]:
load_dotenv()

In [ ]:
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
document = """
Generative AI is a branch of artificial intelligence that can generate
new content such as text, images, audio, video, and code.

Large Language Models are AI models trained on large amounts of text data.
They can understand and generate human-like text.

Retrieval-Augmented Generation, also known as RAG, combines information
retrieval with language generation. RAG retrieves relevant information
from a knowledge base and provides that information to a language model
as context.

The RAG process has three main stages. First, documents are divided into
smaller chunks. Second, the chunks are converted into vector embeddings
and stored in a vector database. Third, when a user asks a question,
relevant chunks are retrieved and provided to the language model.

Vector databases are used to store and search vector embeddings.
FAISS is a library developed by Meta for efficient similarity search
of dense vectors.

Embeddings are numerical representations of text. Text with similar
meaning generally has similar vector representations.

Prompt engineering is the process of designing effective instructions
for a language model. A good prompt can contain system instructions,
context, and the user's question.

RAG is useful for answering questions from private or custom documents
because the model can use information retrieved from those documents.
"""

In [ ]:
def split_text(text, chunk_size=400):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i + chunk_size])
    return chunks


chunks = split_text(document)



In [ ]:
print("Number of chunks:", len(chunks))

In [ ]:

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i + 1}:")
    print(chunk)

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(chunks)

print("Embeddings created.")
print("Shape:", embeddings.shape)

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings.astype("float32"))

print("Embeddings stored in FAISS.")
print("Total vectors:", index.ntotal)

In [ ]:
question = "What is RAG?"

In [ ]:
query_embedding = embedding_model.encode(
    [question]
).astype("float32")

In [ ]:
distances, indices = index.search(
    query_embedding,
    2
)

retrieved_chunks = [
    chunks[i]
    for i in indices[0]
]


In [ ]:
for chunk in retrieved_chunks:
    print(chunk)
    print("-" * 50)

In [ ]:
context = "\n\n".join(retrieved_chunks)

prompt = f"""
Answer the question using the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

print(prompt)

In [ ]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

answer = response.choices[0].message.content

print("Answer:")
print(answer)

In [ ]:
def rag_chat(question):
    query_embedding = embedding_model.encode([question]).astype("float32")

    distances, indices = index.search(
        query_embedding,
        2
    )

    retrieved_chunks = [
        chunks[i]
        for i in indices[0]
    ]
    context = "\n\n".join(retrieved_chunks)
    prompt = f"""
Answer the question using only the provided context.

Context:
{context}

Question:
{question}
Answer:
"""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

In [ ]:
rag_chat("What is RAG?")

## Task 4: Prompt Template For Groq RAG

In [ ]:
def create_rag_prompt(context, question):

    prompt = f"""
System Instructions:
You are a helpful document-based AI assistant.
Answer the user's question using ONLY the provided context.
Do not use outside knowledge.
If the answer is not available in the context,
say "I don't have enough information."

Retrieved Context:
{context}

User Question:
{question}

Answer:
"""

    return prompt

In [ ]:
def rag_chat(question):
    query_embedding = embedding_model.encode(
        [question]
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        2
    )

    retrieved_chunks = [
        chunks[i]
        for i in indices[0]
    ]
    context = "\n\n".join(retrieved_chunks)
    prompt = create_rag_prompt(
        context,
        question
    )

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [ ]:
questions = [
    "What is RAG?",
    "What is FAISS?",
    "What are embeddings?",
    "Why is RAG useful?"
]

for question in questions:

    print("\nQuestion:", question)
    answer = rag_chat(question)
    print("Answer:", answer)

In [ ]:
rag_chat("What is the capital of France?")

# Part 3: Building API using FastAPI

## Task 5: Create FastAPI Application

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

In [ ]:
app = FastAPI(
    title="Groq RAG Chatbot API",
    description="FastAPI backend for Groq-based RAG chatbot",
    version="1.0.0"
)

In [ ]:
@app.get("/health")
def health_check():
    return {
        "status": "healthy"
    }

In [ ]:
class ChatRequest(BaseModel):
    query: str

## Task 6: FastAPI + Groq Integration

In [ ]:
@app.post("/chat")
def chat(request: ChatRequest):
    try:
        answer = rag_chat(request.query)
        return {"answer": answer}
    except Exception as e:
        return {"error": str(e)}

Error handling and not 

In [ ]:
@app.post("/chat")
def chat(request: ChatRequest):

    try:
        if not request.query.strip():
            return {"error": "Query cannot be empty"}
        answer = rag_chat(request.query)
        return {"answer": answer}
    except Exception:
        return {"error": "Unable to generate response"}

# Part 4: Serving the Groq App

## Task 7:Run and Test FastAPI App locally

In [ ]:
import nest_asyncio
import uvicorn

nest_asyncio.apply()

uvicorn.run(
    app,
    host="127.0.0.1",
    port=8000
)

## Task 8: Production Readiness

In [ ]:
requirements = """
groq
sentence-transformers
faiss-cpu
fastapi
uvicorn
pydantic
nest-asyncio
"""

In [ ]:
with open("requirements.txt", "w") as file:
    file.write(requirements.strip())

print("requirements.txt created.")

In [42]:
with open("requirements.txt", "r") as file:
    print(file.read())

groq
sentence-transformers
faiss-cpu
fastapi
uvicorn
pydantic
nest-asyncio


In [ ]:
import logging

logging.basicConfig(level=logging.INFO,format="%(asctime)s - %(levelname)s - %(message)s")

logger = logging.getLogger(__name__)

In [ ]:
@app.post("/chat")
def chat(request: ChatRequest):

    try:
        logger.info("Received chat request")
        if not request.query.strip():
            logger.warning("Empty query received")

            return {"error": "Query cannot be empty"}

        answer = rag_chat(request.query
        )

        logger.info("Chat response generated successfully")

        return {"answer": answer}

    except Exception as e:
        logger.error(f"Error: {str(e)}")
        return {"error": "Unable to generate response"}

## Task 9: End-to-end Demo

In [ ]:
import requests

response = requests.post(
    "http://127.0.0.1:8000/chat",
    json={
        "query": "What is RAG?"
    }
)

print(response.json())

In [ ]:
response = requests.post(
    "http://127.0.0.1:8000/chat",
    json={
        "query": "What are embeddings?"
    }
)

print(response.json())

In [ ]:
response = requests.post(
    "http://127.0.0.1:8000/chat",
    json={
        "query": "What is FAISS?"
    }
)

print(response.json())

In [ ]:
response = requests.post(
    "http://127.0.0.1:8000/chat",
    json={
        "query": "Why is RAG useful?"
    }
)

print(response.json())

In [ ]:
import time
import requests

question = {
    "query": "What is RAG?"
}

start_time = time.time()

response = requests.post(
    "http://127.0.0.1:8000/chat",
    json=question
)

end_time = time.time()

latency = end_time - start_time

print("Response:", response.json())
print(f"Latency: {latency:.2f} seconds")

## Task 10: Observation and Insights

1. Why Groq is suitable for real-time apps

Answer: 
- Groq is suitable for real-time applications because its inference infrastructure is designed for low-latency LLM generation. 
- Fast token generation and streaming can reduce the time users wait for responses, which is useful for chatbots and interactive AI applications.

2. Groq vs OpenAI latency comparison — conceptual

Answer:
- Groq and OpenAI use different infrastructure and offer different models and service configurations, so latency cannot be compared meaningfully using one universal number. 
- Groq is specifically optimized for fast inference, while actual latency for either provider depends on model, input/output token count, network conditions, and service load. 
- A fair comparison should benchmark the same task, similar model size, similar prompt/output lengths, and the same client/network conditions.

3. Benefits of API-first GenAI architecture

Answer:
- An API-first GenAI architecture separates the AI backend from the frontend. This allows multiple clients such as web applications, mobile applications, and other services to use the same AI API. 
- It also makes testing, authentication, monitoring, and future frontend integration easier.